# DevAgent 项目 - Day 8：Human-in-the-Loop + 增强 Reflection + 状态持久化

目标：
- 实现 Human-in-the-Loop（关键操作人工确认）
- 升级为 LLM-based Reflection（智能自我反思）
- 添加状态持久化（SQLiteSaver）
- 提升系统可控性和生产级能力

In [2]:
# ==================== 2. 配置环境 ====================
import os
from dotenv import load_dotenv
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage
from typing import TypedDict, Annotated, List, Sequence
import operator

from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.checkpoint.redis import RedisSaver
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import HumanMessage

load_dotenv(override=True)

llm = ChatOllama(
    model="qwen2.5:14b",
    temperature=0.2,
    num_ctx=8192,
    num_gpu=999,
    base_url="http://host.docker.internal:11434"
)

print("✅ LLM 初始化完成")

✅ LLM 初始化完成


In [8]:
from langgraph.checkpoint.redis import RedisSaver
import redis

# ==================== 测试连接 ====================
test_urls = [
    "redis://host.docker.internal:6379",     # Mac/Windows
    "redis://172.17.0.1:6379",               # Docker 默认网桥
    "redis://localhost:6379",                # 如果是 host network
]

for url in test_urls:
    print(f"\n🔍 测试连接: {url}")
    try:
        # 先用 redis-py 测试基础连接
        r = redis.from_url(url, socket_connect_timeout=3)
        r.ping()
        print("✅ 连接成功！")
        
        # 如果成功，则用于 LangGraph
        checkpointer = RedisSaver.from_conn_string(url)
        print("✅ RedisSaver 初始化成功")
        break
    except Exception as e:
        print(f"❌ 连接失败: {e}")


🔍 测试连接: redis://host.docker.internal:6379
✅ 连接成功！
✅ RedisSaver 初始化成功


In [9]:
# ==================== Day 8 核心演示：持久化 + Human-in-the-Loop ====================

from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import HumanMessage, BaseMessage
from typing import TypedDict, Annotated, List, Sequence
import operator

class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add]
    next: str
    reflections: List[str]
    final_answer: str
    human_feedback: str

# ==================== 1. 状态持久化 (SqliteSaver) ====================
print("✅ 状态持久化数据库已启用：devagent_checkpoints.db")

# ==================== 2. Human-in-the-Loop Node ====================
def human_input_node(state: AgentState):
    print("\n" + "="*80)
    print("🛑 【Human-in-the-Loop】系统需要人工确认")
    print("="*80)
    
    last_msg = state["messages"][-1].content if state["messages"] else ""
    print("当前系统输出预览：")
    print(last_msg[:600] + "..." if len(last_msg) > 600 else last_msg)
    
    feedback = input("\n请输入您的意见或修改建议（直接回车继续）：\n> ").strip()
    
    if feedback:
        print(f"✅ 已接收用户反馈: {feedback[:100]}...")
        return {
            "human_feedback": feedback,
            "messages": state["messages"] + [HumanMessage(content=f"[用户反馈]: {feedback}")]
        }
    return {"human_feedback": ""}

# ==================== 3. 其他简单节点 ====================
def researcher_node(state):
    print("▶️ Researcher 执行中...")
    return {"messages": state["messages"] + [HumanMessage(content="【Researcher】已完成简历分析")]}

def reflection_node(state):
    print("🤔 Reflection 执行中...")
    return {
        "reflections": ["反思：输出已覆盖核心点，可增加更多量化案例。"],
        "messages": state["messages"]
    }

def final_answer_node(state):
    final_text = state["messages"][-1].content
    if state.get("reflections"):
        final_text += "\n\n【系统反思】\n" + "\n".join(state["reflections"])
    if state.get("human_feedback"):
        final_text += f"\n\n【用户反馈】：{state['human_feedback']}"
    
    return {"final_answer": final_text, "messages": state["messages"]}

# ==================== 4. 构建 Graph ====================
workflow = StateGraph(AgentState)

workflow.add_node("Researcher", researcher_node)
workflow.add_node("Reflection", reflection_node)
workflow.add_node("HumanInput", human_input_node)
workflow.add_node("Final_Answer", final_answer_node)

workflow.add_edge(START, "Researcher")
workflow.add_edge("Researcher", "Reflection")
workflow.add_edge("Reflection", "HumanInput")     # 关键：反思后进行人工确认
workflow.add_edge("HumanInput", "Final_Answer")
workflow.add_edge("Final_Answer", END)



✅ 状态持久化数据库已启用：devagent_checkpoints.db


In [ ]:
# 关键：传入 checkpointer
with SqliteSaver.from_conn_string("devagent_checkpoints.db") as checkpointer:
    multi_agent = workflow.compile(checkpointer=checkpointer)

    # ==================== 5. 测试（体现两个功能） ====================
    config = {"configurable": {"thread_id": "demo_session_001"}}

    result = multi_agent.invoke({
        "messages": [HumanMessage(content="帮我准备 AI Engineer 岗位的简历优化建议")],
        "reflections": [],
        "human_feedback": ""
    }, config=config)

    print("\n" + "="*60)
    print("🎯 最终输出：", result.get("final_answer"))
    print("🎯 人工干预", result.get("human_feedback"))
    print("🎯 自我反思", result.get("reflections"))

▶️ Researcher 执行中...
🤔 Reflection 执行中...

🛑 【Human-in-the-Loop】系统需要人工确认
当前系统输出预览：
【Researcher】已完成简历分析
✅ 已接收用户反馈: 帮我看下这个简历...

🎯 最终输出： [用户反馈]: 帮我看下这个简历

【系统反思】
反思：输出已覆盖核心点，可增加更多量化案例。

【用户反馈】：帮我看下这个简历
🎯 人工干预 帮我看下这个简历
🎯 自我反思 ['反思：输出已覆盖核心点，可增加更多量化案例。']


在这里重启kernel然后执行下面操作，同一个treadid会保存上次的所有agentstate

In [ ]:
# 关键：传入 checkpointer
with SqliteSaver.from_conn_string("devagent_checkpoints.db") as checkpointer:
    multi_agent = workflow.compile(checkpointer=checkpointer)

    # ==================== 5. 测试（体现两个功能） ====================
    config = {"configurable": {"thread_id": "demo_session_001"}}

    result = multi_agent.invoke({
        "messages": [HumanMessage(content="在帮我看一次更新后的简历")],
        "reflections": [],
        "human_feedback": ""
    }, config=config)

    print("\n" + "="*60)
    print("🎯 最终输出：", result.get("final_answer"))
    print("🎯 人工干预", result.get("human_feedback"))
    print("🎯 自我反思", result.get("reflections"))

▶️ Researcher 执行中...
🤔 Reflection 执行中...

🛑 【Human-in-the-Loop】系统需要人工确认
当前系统输出预览：
【Researcher】已完成简历分析
✅ 已接收用户反馈: 在帮我看下...


In [10]:
with RedisSaver.from_conn_string(os.getenv("REDIS_URL")) as checkpointer:
    multi_agent = workflow.compile(checkpointer=checkpointer)

    # ==================== 5. 测试（体现两个功能） ====================
    config = {"configurable": {"thread_id": "demo_session_001"}}

    result = multi_agent.invoke({
        "messages": [HumanMessage(content="在帮我看一次更新后的简历")],
        "reflections": [],
        "human_feedback": ""
    }, config=config)

    print("\n" + "="*60)
    print("🎯 最终输出：", result.get("final_answer"))
    print("🎯 人工干预", result.get("human_feedback"))
    print("🎯 自我反思", result.get("reflections"))

▶️ Researcher 执行中...
🤔 Reflection 执行中...

🛑 【Human-in-the-Loop】系统需要人工确认
当前系统输出预览：
【Researcher】已完成简历分析


✅ 已接收用户反馈: 帮我再看下...

🎯 最终输出： [用户反馈]: 帮我再看下

【系统反思】
反思：输出已覆盖核心点，可增加更多量化案例。

【用户反馈】：帮我再看下
🎯 人工干预 帮我再看下
🎯 自我反思 ['反思：输出已覆盖核心点，可增加更多量化案例。']


In [11]:
import redis
import json

# 连接 Redis
r = redis.from_url("redis://host.docker.internal:6379", decode_responses=True)

# 查询某个 thread 的所有 checkpoint
thread_id = "demo_session_001"

print(f"=== 查询 thread_id: {thread_id} 的 Checkpoint 数据 ===\n")

# 查找所有相关 key
keys = r.keys(f"checkpoint:{thread_id}*")
print(f"找到 {len(keys)} 个相关 key:\n")

for key in sorted(keys):
    data = r.get(key)
    print(f"Key: {key}")
    try:
        parsed = json.loads(data)
        print(json.dumps(parsed, indent=2, ensure_ascii=False)[:1000] + "..." if len(data) > 1000 else json.dumps(parsed, indent=2, ensure_ascii=False))
    except:
        print(data[:500] + "..." if len(data) > 500 else data)
    print("-" * 80)

=== 查询 thread_id: demo_session_001 的 Checkpoint 数据 ===

找到 6 个相关 key:



ResponseError: WRONGTYPE Operation against a key holding the wrong kind of value